In [7]:
import numpy as np
import pandas as pd
from pathlib import Path

# -------------------------------------------------------------------
# 1. Load data
# -------------------------------------------------------------------
input_path = "RCC_cleaned_2018.csv"   # change if your file is elsewhere
df = pd.read_csv(input_path)

# -------------------------------------------------------------------
# 2. Create numeric survival time and event indicator
# -------------------------------------------------------------------
# Survival months: convert to numeric, set "Unknown" to NaN
df["Survival_months_num"] = pd.to_numeric(
    df["Survival months"].replace("Unknown", np.nan)
)

# Drop rows with unknown survival time (cannot use them for time-based labels)
df = df[df["Survival_months_num"].notna()].copy()

# Event: 1 = Dead, 0 = Alive
df["event"] = (df["Vital status recode (study cutoff used)"] == "Dead").astype(int)

# For information only (not strictly needed for the labels)
df["time_years"] = df["Survival_months_num"] / 12.0

print("After cleaning:")
print(df[["Survival months", "Survival_months_num",
          "Vital status recode (study cutoff used)", "event"]].head())
print("\nMax follow-up (months):", df["Survival_months_num"].max())

# -------------------------------------------------------------------
# 3. Function to build horizon-specific datasets
# -------------------------------------------------------------------
def make_horizon_dataset(df, t_years):
    """
    Build a binary classification dataset for 'death within t_years'.

    Positive class (1): died on/before t_years.
    Negative class (0): alive and followed longer than t_years.
    Censored before t_years are excluded.
    """
    t_months = t_years * 12

    # Keep only patients with clear status at or beyond t
    mask_clear = (
        ((df["Survival_months_num"] <= t_months) & (df["event"] == 1)) |
        (df["Survival_months_num"] > t_months)
    )

    d = df.loc[mask_clear].copy()

    label_col = f"y_{t_years}yr"
    d[label_col] = (
        (d["Survival_months_num"] <= t_months) & (d["event"] == 1)
    ).astype(int)

    return d, label_col

# -------------------------------------------------------------------
# 4. Build 1–4 year datasets and save
# -------------------------------------------------------------------
output_dir = Path("classification_datasets")
output_dir.mkdir(exist_ok=True)

# columns we probably don't want as predictors
meta_cols = [
    "Survival months",
    "Survival_months_num",
    "time_years",
    "event",
    "Vital status recode (study cutoff used)", 'Sequence number', 'Histology recode - broad groupings', 'AYA site recode 2020 Revision','COD to site recode'
,'COD to site recode','Vital status recode (study cutoff used)','Diagnostic Confirmation','PRCDA 2020','RX Summ--Surg Prim Site (1998+)','Year of diagnosis'
]

summary = []

for t in [1, 2, 3, 4, 5]:
    d, label_col = make_horizon_dataset(df, t)

    # skip horizons where only one class exists (e.g. 5-year in this dataset)
    if d[label_col].nunique() < 2:
        print(f"Skipping {t}-year horizon: only one class present.")
        continue

    # Drop meta columns; keep original features + label
    dataset = d.drop(columns=meta_cols, errors="ignore")

    out_path = output_dir / f"RCC_{t}yr_mortality_dataset.csv"
    dataset.to_csv(out_path, index=False)

    event_rate = dataset[label_col].mean()
    summary.append((t, dataset.shape[0], event_rate))

    print(
        f"{t}-year dataset: n = {dataset.shape[0]}, "
        f"event_rate = {event_rate:.3f}  -> saved to {out_path}"
    )

print("\nSummary (t_years, n_samples, event_rate):")
for t, n, r in summary:
    print(f"{t}-year: n={n}, event_rate={r:.3f}")


After cleaning:
  Survival months  Survival_months_num  \
0               9                  9.0   
1              33                 33.0   
2              16                 16.0   
3               1                  1.0   
4              11                 11.0   

  Vital status recode (study cutoff used)  event  
0                                   Alive      0  
1                                   Alive      0  
2                                   Alive      0  
3                                    Dead      1  
4                                   Alive      0  

Max follow-up (months): 59.0
1-year dataset: n = 18644, event_rate = 0.145  -> saved to classification_datasets\RCC_1yr_mortality_dataset.csv
2-year dataset: n = 14370, event_rate = 0.246  -> saved to classification_datasets\RCC_2yr_mortality_dataset.csv
3-year dataset: n = 10930, event_rate = 0.369  -> saved to classification_datasets\RCC_3yr_mortality_dataset.csv
4-year dataset: n = 7309, event_rate = 0.590  -> saved t

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# -------------------------------------------------------------------
# 1. Load data
# -------------------------------------------------------------------
input_path = "RCC_cleaned_2018.csv"   # change if your file is elsewhere
df = pd.read_csv(input_path)
df.shape

(24060, 43)

In [8]:
dataset.shape

(7309, 34)